# 01 - Cleaning: IBM Employee Attrition Dataset
Central command notebook. Cleans raw_dirty.csv, saves clean_master.csv, then splits into train/test/present.

In [3]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../src')
from utils import basic_report, TARGET

df = pd.read_csv('/content/IBM_HR_Attrition_Dirty.csv')
basic_report(df)
df.head()


shape: (1515, 35)

nulls:
 Age                  77
TotalWorkingYears    62
dtype: int64

dupes: 29

dtypes:
 Age                         float64
Attrition                    object
BusinessTravel               object
DailyRate                     int64
Department                   object
DistanceFromHome              int64
Education                     int64
EducationField               object
EmployeeCount                 int64
EmployeeNumber                int64
EnvironmentSatisfaction       int64
Gender                       object
HourlyRate                    int64
JobInvolvement                int64
JobLevel                      int64
JobRole                      object
JobSatisfaction               int64
MaritalStatus                object
MonthlyIncome                object
MonthlyRate                   int64
NumCompaniesWorked            int64
Over18                       object
OverTime                     object
PercentSalaryHike             int64
PerformanceRating          

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41.0,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8.0,0,1,6,4,0,5
1,49.0,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10.0,3,3,10,7,1,7
2,37.0,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7.0,3,3,0,0,0,0
3,33.0,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8.0,3,3,8,7,3,0
4,27.0,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6.0,3,3,2,2,2,2


## 1. Fix column names & whitespace

In [4]:
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()


## 2. Drop exact duplicate rows

In [5]:
before = len(df)
df = df.drop_duplicates()
print(f"dropped {before - len(df)} duplicate rows")


dropped 29 duplicate rows


## 3. Standardize inconsistent categorical casing/labels

In [6]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.title()

# common attrition dataset fixes - adjust to whatever dirt you introduced
if TARGET in df.columns:
    df[TARGET] = df[TARGET].replace({'Y': 'Yes', 'N': 'No', '1': 'Yes', '0': 'No'})


## 4. Handle missing values

In [7]:
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include='object').columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df.isnull().sum().sum()  # should be 0


np.int64(0)

## 5. Fix dtypes (e.g. numbers stored as strings)

In [8]:
for col in df.columns:
    if df[col].dtype == 'object':
        converted = pd.to_numeric(df[col], errors='coerce')
        if converted.notnull().mean() > 0.95:  # mostly numeric -> convert
            df[col] = converted.fillna(converted.median())


## 6. Outlier check (cap using IQR)

In [9]:
num_cols = df.select_dtypes(include=np.number).columns
for col in num_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    df[col] = df[col].clip(lower, upper)


## 7. Final check & save clean_master.csv

In [11]:
basic_report(df)
df.to_csv('clean_master.csv', index=False)


shape: (1486, 35)

nulls:
 Series([], dtype: int64)

dupes: 3

dtypes:
 Age                         float64
Attrition                    object
BusinessTravel               object
DailyRate                     int64
Department                   object
DistanceFromHome              int64
Education                     int64
EducationField               object
EmployeeCount                 int64
EmployeeNumber                int64
EnvironmentSatisfaction       int64
Gender                       object
HourlyRate                    int64
JobInvolvement                int64
JobLevel                      int64
JobRole                      object
JobSatisfaction               int64
MaritalStatus                object
MonthlyIncome                object
MonthlyRate                   int64
NumCompaniesWorked          float64
Over18                       object
OverTime                     object
PercentSalaryHike             int64
PerformanceRating             int64
RelationshipSatisfaction    

## 8. Split into train / test / present
Present set drops the target column - simulates unseen new employees.

In [12]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df[TARGET] if TARGET in df else None)
test_df, present_df = train_test_split(temp_df, test_size=0.5, random_state=42)

present_df = present_df.drop(columns=[TARGET]) if TARGET in present_df.columns else present_df

train_df.to_csv('train.csv', index=False)
test_df.to_csv('test.csv', index=False)
present_df.to_csv('present.csv', index=False)

print(train_df.shape, test_df.shape, present_df.shape)


(1040, 35) (223, 35) (223, 34)
